# 문제 3 — TF(Task Force) 최적 인원 구성
**입력 파일 3개**
- `RND_HR_DB.xlsx` → 전체 인사DB + 역량점수
- `Project_Required_Skills_Matrix.xlsx` → 기존 20개 과제 스킬 매핑
- `TF_Requirements.xlsx` → TF 필수 스킬 + 목표 인원

**핵심 원칙**
- 기존 과제 소속 인원 중에서 TF로 차출
- 차출 후 기존 팀에 공백이 생기면 안 됨
- 세 파일 모두 코드와 같은 폴더에 위치

**흐름**
```
1단계: TF 정보 확인 + 기존 팀 현황 EDA + 차출 허용 팀/상한 설정
2단계: 차출 후보 개인별 적합도 산출 (카드 1~4)
3단계: 공백 방지 옵션 + ILP 최적 TF 구성
```

## 0. 라이브러리 임포트

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import ipywidgets as widgets
from IPython.display import display
import warnings
warnings.filterwarnings('ignore')

plt.rcParams['font.family'] = 'Malgun Gothic'
plt.rcParams['axes.unicode_minus'] = False
print('라이브러리 로드 완료')

## 1. 데이터 로드

In [ ]:
HR_FILE   = 'RND_HR_DB.xlsx'
MAT_FILE  = 'Project_Required_Skills_Matrix.xlsx'
TF_FILE   = 'TF_Requirements.xlsx'

# ── 인사 DB ───────────────────────────────────────────────────────────────────
df_hr    = pd.read_excel(HR_FILE, sheet_name='인사_DB')
df_skill = pd.read_excel(HR_FILE, sheet_name='R&D_역량_점수')

# ── 기존 과제-스킬 매핑 ───────────────────────────────────────────────────────
df_mat   = pd.read_excel(MAT_FILE, sheet_name='역량_요구_매트릭스')
mat_idx  = df_mat.set_index('필수 요소기술')
proj_skill_bin = (mat_idx == '●').astype(int)
ALL_PROJECTS   = [c for c in df_mat.columns if c != '필수 요소기술']

# ── TF 요구사항 ───────────────────────────────────────────────────────────────
tf_info  = pd.read_excel(TF_FILE, sheet_name='TF_기본정보')
tf_skills_df = pd.read_excel(TF_FILE, sheet_name='TF_필수스킬')

TF_CODE   = tf_info['과제코드'].iloc[0]
TF_NAME   = tf_info['과제명'].iloc[0]
TF_SIZE   = int(tf_info['목표인원'].iloc[0])
TF_SKILLS = tf_skills_df['필수스킬'].tolist()

# ── 스킬 wide matrix ──────────────────────────────────────────────────────────
df_wide   = df_skill.pivot_table(
    index='사번', columns='요소기술', values='기술 레벨 (Score)', aggfunc='max').fillna(0)
ALL_SKILLS = sorted(df_wide.columns.tolist())
N_TOTAL    = len(df_wide)

# 스킬 메타
holders_all  = (df_wide > 0).sum()
holders_lv4  = (df_wide >= 4).sum()

print(f'TF 과제: {TF_CODE} ({TF_NAME})')
print(f'TF 목표 인원: {TF_SIZE}명')
print(f'TF 필수 스킬 ({len(TF_SKILLS)}개): {TF_SKILLS}')
print(f'전체 인원: {N_TOTAL}명 | 기존 과제: {len(ALL_PROJECTS)}개')

## 2. 전처리

In [ ]:
# ── 팀별 기존 기여분 계산 (공백 방지 제약용 고정 상수) ─────────────────────
rmap   = {'사원':1,'대리':2,'과장':3,'차장':4,'부장':5}
df_hr['직급수'] = df_hr['직위'].map(rmap)
pool_f = (df_hr['성별']=='여').mean()
pool_r = df_hr['직급수'].mean()

team_members     = {}   # 과제 → 사번 리스트
team_skill_sum   = {}   # (과제, 스킬) → 기존 팀원 레벨 합
team_skill_hold  = {}   # (과제, 스킬) → 기존 팀원 보유자 수
team_fem_count   = {}   # 과제 → 기존 팀원 여성 수
team_rank_sum    = {}   # 과제 → 기존 팀원 직급 합

for p in ALL_PROJECTS:
    members = df_hr[df_hr['소속과제명']==p]['사번'].tolist()
    team_members[p] = members
    mbr_wide = df_wide.loc[df_wide.index.isin(members)]
    mbr_hr   = df_hr[df_hr['소속과제명']==p]

    for s in ALL_SKILLS:
        team_skill_sum[(p,s)]  = float(mbr_wide[s].sum()) if s in mbr_wide.columns else 0.0
        team_skill_hold[(p,s)] = int((mbr_wide[s]>0).sum()) if s in mbr_wide.columns else 0

    team_fem_count[p]  = int((mbr_hr['성별']=='여').sum())
    team_rank_sum[p]   = float(mbr_hr['직급수'].sum())

Nj = {p: len(team_members[p]) for p in ALL_PROJECTS}   # 팀별 현재 인원

# ── TF 스킬별 전체 보유자 현황 ────────────────────────────────────────────────
tf_skill_supply = {}
for s in TF_SKILLS:
    tf_skill_supply[s] = int((df_wide[s]>0).sum()) if s in df_wide.columns else 0

print('전처리 완료')
print(f'\nTF 필수 스킬별 전체 보유자:')
for s,n in tf_skill_supply.items():
    print(f'  {s}: {n}명')

## 1단계 — TF 정보 확인 + 기존 팀 현황 EDA
> 차출을 허용할 팀과 팀별 최대 차출 인원을 설정합니다.

In [ ]:
# ── TF 스킬 커버리지 현황 ────────────────────────────────────────────────────
AVG_LEVEL = 2.8

# 팀별 TF 스킬 보유 현황 (차출 후보 파악용)
team_stats = []
for p in ALL_PROJECTS:
    n = Nj[p]
    n_f = team_fem_count[p]
    avg_rk = team_rank_sum[p] / n if n > 0 else 0

    # TF 필수 스킬 보유자 수 (이 팀에서 TF 스킬 가진 사람)
    tf_skill_holders_in_team = {}
    for s in TF_SKILLS:
        mbr_wide = df_wide.loc[df_wide.index.isin(team_members[p])]
        tf_skill_holders_in_team[s] = int((mbr_wide[s]>0).sum()) if s in mbr_wide.columns else 0
    total_tf_holders = sum(tf_skill_holders_in_team.values())

    # 기존 과제 스킬 갭
    req_s = proj_skill_bin.index[proj_skill_bin[p]==1].tolist() if p in proj_skill_bin.columns else []
    n_gap = sum(1 for s in req_s if max(0, AVG_LEVEL*n - team_skill_sum.get((p,s),0)) > 0)

    team_stats.append({
        '과제': p, '현재인원': n,
        'TF스킬보유자합': total_tf_holders,
        '현재스킬갭수': n_gap,
        '여성비율(%)': round(n_f/n*100 if n>0 else 0, 1),
        '평균직급': round(avg_rk, 2)
    })

team_stats_df = pd.DataFrame(team_stats)
print('[기존 팀 현황 — TF 스킬 보유자 많을수록 차출 가능성 높음]')
display(team_stats_df)

In [ ]:
# ── 시각화: 팀별 TF 스킬 보유자 수 ──────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(16, 5))

axes[0].bar(team_stats_df['과제'], team_stats_df['TF스킬보유자합'], color='#6366f1aa')
axes[0].set_title('팀별 TF 필수 스킬 보유자 합산', fontsize=11)
axes[0].set_xlabel('과제'); axes[0].tick_params(axis='x', rotation=45)

axes[1].bar(team_stats_df['과제'], team_stats_df['현재스킬갭수'], color='#ef4444aa')
axes[1].set_title(f'팀별 현재 스킬 갭 수 (AVG_LEVEL={AVG_LEVEL} 기준)', fontsize=11)
axes[1].set_xlabel('과제'); axes[1].tick_params(axis='x', rotation=45)

plt.tight_layout(); plt.show()

### 차출 허용 팀 선택

In [ ]:
cb_allow = {}; rows_allow = []
for p in ALL_PROJECTS:
    row = team_stats_df[team_stats_df['과제']==p].iloc[0]
    label = (f"{p}  │  현재 {row['현재인원']}명  │  "
             f"TF스킬보유자 {row['TF스킬보유자합']}명  │  "
             f"스킬갭 {row['현재스킬갭수']}개  │  "
             f"여성 {row['여성비율(%)']}%  │  평균직급 {row['평균직급']}")
    cb = widgets.Checkbox(value=True, description=label,
                          layout=widgets.Layout(width='720px'),
                          style={'description_width':'initial'})
    cb_allow[p] = cb; rows_allow.append(cb)

btn_all_on  = widgets.Button(description='전체 허용', button_style='primary', layout=widgets.Layout(width='120px'))
btn_all_off = widgets.Button(description='전체 해제', button_style='',        layout=widgets.Layout(width='120px'))
def all_on(_):
    for cb in cb_allow.values(): cb.value = True
def all_off(_):
    for cb in cb_allow.values(): cb.value = False
btn_all_on.on_click(all_on); btn_all_off.on_click(all_off)

header_a = widgets.HTML('<b>차출을 허용할 팀 선택 (체크 = 허용)</b><br><br>')
display(widgets.VBox([header_a, widgets.HBox([btn_all_on, btn_all_off]),
                      widgets.VBox(rows_allow, layout=widgets.Layout(
                          border='1px solid #ccc', padding='8px'))]))

### 팀별 최대 차출 인원 설정

In [ ]:
default_max = widgets.BoundedIntText(
    value=3, min=0, max=20, description='팀당 기본 상한:',
    style={'description_width':'initial'}, layout=widgets.Layout(width='250px'))
apply_btn = widgets.Button(description='기본값 적용', button_style='primary',
                            layout=widgets.Layout(width='120px'))

max_out = {}; max_rows = []
for p in ALL_PROJECTS:
    inp = widgets.BoundedIntText(value=3, min=0, max=20, description=f'{p} 최대:',
                                  style={'description_width':'initial'},
                                  layout=widgets.Layout(width='180px'))
    max_out[p] = inp; max_rows.append(inp)

def apply_default(_):
    for inp in max_out.values(): inp.value = default_max.value
apply_btn.on_click(apply_default)

header_m = widgets.HTML('<b>팀별 최대 차출 인원 설정</b><br><br>')
grid_rows = [widgets.HBox(max_rows[i:i+4]) for i in range(0, len(max_rows), 4)]
display(widgets.VBox([header_m, widgets.HBox([default_max, apply_btn]),
                      widgets.VBox(grid_rows, layout=widgets.Layout(
                          border='1px solid #ccc', padding='8px'))]))

## 2단계 — 차출 후보 개인별 적합도 산출
> **위 1단계 설정 완료 후 이 셀부터 실행하세요.**

In [ ]:
# ── 차출 허용 팀 + 후보 인원 확정 ──────────────────────────────────────────
allowed_projects = [p for p, cb in cb_allow.items() if cb.value]
MAX_OUT = {p: max_out[p].value for p in ALL_PROJECTS}

# 차출 후보 = 허용된 팀 소속 전원
candidate_ids = []
for p in allowed_projects:
    candidate_ids.extend(team_members[p])
candidate_ids = list(dict.fromkeys(candidate_ids))  # 중복 제거

cand_wide = df_wide.loc[df_wide.index.isin(candidate_ids)]
cand_hr   = df_hr[df_hr['사번'].isin(candidate_ids)].reset_index(drop=True)
N_CAND    = len(candidate_ids)

print(f'차출 허용 팀 ({len(allowed_projects)}개): {allowed_projects}')
print(f'차출 후보 인원: {N_CAND}명')
print(f'TF 목표 인원: {TF_SIZE}명')
if sum(MAX_OUT[p] for p in allowed_projects) < TF_SIZE:
    print(f'⚠️  총 차출 상한 합계({sum(MAX_OUT[p] for p in allowed_projects)}명) < TF 목표({TF_SIZE}명)')
    print('   팀별 상한을 늘리거나 허용 팀을 추가하세요.')
else:
    print(f'✅ 총 차출 상한 합계({sum(MAX_OUT[p] for p in allowed_projects)}명) ≥ TF 목표({TF_SIZE}명)')

In [ ]:
# ── IDF 계산 (전체 기준) ─────────────────────────────────────────────────────
idf_raw  = np.log((N_TOTAL + 1) / (holders_all.reindex(ALL_SKILLS).fillna(0) + 1))
idf_norm = ((idf_raw - idf_raw.min()) / (idf_raw.max() - idf_raw.min())).round(4)

df_idf = pd.DataFrame({
    '스킬': ALL_SKILLS,
    '전체보유자': holders_all.reindex(ALL_SKILLS).fillna(0).astype(int),
    'IDF_norm': idf_norm.reindex(ALL_SKILLS)
}).sort_values('IDF_norm', ascending=False).reset_index(drop=True)

q67 = df_idf['IDF_norm'].quantile(0.67); q33 = df_idf['IDF_norm'].quantile(0.33)
def idf_class(v):
    return '희귀' if v >= q67 else ('보편' if v <= q33 else '보통')
df_idf['분류'] = df_idf['IDF_norm'].apply(idf_class)

# ── 카드 2 (문제 3 전용): TF 스킬별 차출 후보 보유 현황 ───────────────────────
# P21이 단일 과제 → KSS 대신 "차출 후보 중 보유자 수"로 공급 현황 표시
tf_supply_cand = {}
for s in TF_SKILLS:
    tf_supply_cand[s] = int((cand_wide[s]>0).sum()) if s in cand_wide.columns else 0

df_tf_supply = pd.DataFrame({
    'TF스킬': TF_SKILLS,
    '전체보유자': [tf_skill_supply[s] for s in TF_SKILLS],
    '후보보유자': [tf_supply_cand[s] for s in TF_SKILLS],
}).sort_values('후보보유자')
df_tf_supply['후보보유율(%)'] = (df_tf_supply['후보보유자'] / N_CAND * 100).round(1)

# KSS_norm 대용: 후보 보유자 수 적을수록 희귀한 공급
max_sup = df_tf_supply['후보보유자'].max()
min_sup = df_tf_supply['후보보유자'].min()
df_tf_supply['공급희귀도'] = ((max_sup - df_tf_supply['후보보유자']) /
                              (max_sup - min_sup + 1e-9)).round(4)

# 스킬 난이도
difficulty_raw = pd.Series({
    s: (1 - holders_lv4.get(s,0)/holders_all.get(s,1)) if holders_all.get(s,0)>0 else 0.0
    for s in ALL_SKILLS})
df_diff = pd.DataFrame({
    '스킬': ALL_SKILLS,
    '전체보유자': holders_all.reindex(ALL_SKILLS).fillna(0).astype(int),
    '레벨4이상': holders_lv4.reindex(ALL_SKILLS).fillna(0).astype(int),
    '난이도': difficulty_raw
}).sort_values('난이도', ascending=False).reset_index(drop=True)

print('스킬 메타 계산 완료')
print('\n[TF 스킬별 차출 후보 보유 현황]')
display(df_tf_supply)

## EDA — 카드 1 : 스킬 희귀도 (IDF, 전체 기준)

In [ ]:
TOP_N = 15
top_rare   = df_idf.head(TOP_N)
top_common = df_idf.tail(TOP_N).iloc[::-1]
color_map  = {'희귀':'#E05C5C','보통':'#7A9CC6','보편':'#6DBF8A'}

fig, axes = plt.subplots(1, 2, figsize=(16, 6))
fig.suptitle('카드 1 : 스킬 희귀도 (IDF_norm, 전체 기준)', fontsize=13, fontweight='bold', y=1.01)
for ax, df_sub, title in [
    (axes[0], top_rare,   f'희귀 스킬 TOP {TOP_N}'),
    (axes[1], top_common, f'보편 스킬 TOP {TOP_N}'),
]:
    colors = [color_map[c] for c in df_sub['분류']]
    bars = ax.barh(df_sub['스킬'], df_sub['IDF_norm'], color=colors)
    for bar, h in zip(bars, df_sub['전체보유자']):
        ax.text(bar.get_width()+0.005, bar.get_y()+bar.get_height()/2,
                f'{h}명', va='center', fontsize=8)
    ax.set_xlim(0,1.15); ax.set_xlabel('IDF_norm'); ax.set_title(title, fontsize=11); ax.invert_yaxis()
# TF 스킬 강조
for ax in axes:
    for label in ax.get_yticklabels():
        if label.get_text() in TF_SKILLS:
            label.set_color('#f59e0b'); label.set_fontweight('bold')
patches = [mpatches.Patch(color=v,label=k) for k,v in color_map.items()]
fig.legend(handles=patches, loc='lower center', ncol=3, bbox_to_anchor=(0.5,-0.05))
plt.tight_layout(); plt.show()

### 카드 1 선택 — 희귀도를 적합도에 반영할 스킬

In [ ]:
cb_idf = {}; rows_idf = []
for _, row in df_idf.iterrows():
    s = row['스킬']
    is_tf = '⭐TF필수' if s in TF_SKILLS else ''
    label = f"{s}  │  IDF={row['IDF_norm']:.3f}  │  전체보유자={row['전체보유자']}명  │  [{row['분류']}]  {is_tf}"
    cb = widgets.Checkbox(value=False, description=label,
                          layout=widgets.Layout(width='600px'),
                          style={'description_width':'initial'})
    cb_idf[s] = cb; rows_idf.append(cb)

btn_rare = widgets.Button(description='희귀 전체 선택', button_style='danger',  layout=widgets.Layout(width='140px'))
btn_tf1  = widgets.Button(description='TF 스킬 선택',  button_style='warning', layout=widgets.Layout(width='130px'))
btn_clr1 = widgets.Button(description='전체 해제',      button_style='',        layout=widgets.Layout(width='120px'))
def sel_rare(_):
    for s,cb in cb_idf.items():
        cb.value = df_idf[df_idf['스킬']==s]['분류'].iloc[0]=='희귀'
def sel_tf1(_):
    for s,cb in cb_idf.items(): cb.value = s in TF_SKILLS
def clr1(_):
    for cb in cb_idf.values(): cb.value=False
btn_rare.on_click(sel_rare); btn_tf1.on_click(sel_tf1); btn_clr1.on_click(clr1)
cnt1 = widgets.HTML('')
def upd1(change): cnt1.value = f'<b>{sum(cb.value for cb in cb_idf.values())}개 선택됨</b>'
for cb in cb_idf.values(): cb.observe(upd1, names='value')
display(widgets.VBox([
    widgets.HTML('<b>희귀도를 적합도에 반영할 스킬 선택 (I_idf=1)</b><br>⭐ = TF 필수 스킬<br>'),
    widgets.HBox([btn_rare, btn_tf1, btn_clr1, cnt1]),
    widgets.VBox(rows_idf, layout=widgets.Layout(height='300px', overflow_y='scroll',
                                                   border='1px solid #ccc', padding='6px'))]))

## EDA — 카드 2 : TF 스킬별 차출 후보 공급 현황 (문제 3 전용)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 5))
fig.suptitle('카드 2 : TF 스킬별 차출 후보 보유 현황', fontsize=13, fontweight='bold')

colors_supply = ['#ef4444aa' if v < TF_SIZE else '#6366f1aa'
                 for v in df_tf_supply['후보보유자']]
bars = axes[0].barh(df_tf_supply['TF스킬'], df_tf_supply['후보보유자'], color=colors_supply)
for bar, row in zip(bars, df_tf_supply.itertuples()):
    axes[0].text(bar.get_width()+0.3, bar.get_y()+bar.get_height()/2,
                 f'{row.후보보유자}명 ({row.후보보유율}%)', va='center', fontsize=9)
axes[0].axvline(TF_SIZE, color='red', linestyle='--', alpha=0.7, label=f'TF 목표 {TF_SIZE}명')
axes[0].set_xlabel('차출 후보 중 보유자 수')
axes[0].set_title('TF 스킬별 차출 후보 보유자 수', fontsize=11)
axes[0].legend(fontsize=9); axes[0].invert_yaxis()

bars2 = axes[1].barh(df_tf_supply['TF스킬'], df_tf_supply['공급희귀도'],
                      color='#f59e0baa')
axes[1].set_xlabel('공급 희귀도 (높을수록 보유자 적음)')
axes[1].set_title('TF 스킬별 공급 희귀도', fontsize=11); axes[1].invert_yaxis()

plt.tight_layout(); plt.show()

shortage = df_tf_supply[df_tf_supply['후보보유자'] < TF_SIZE]
if not shortage.empty:
    print(f'⚠️  차출 후보 보유자가 TF 목표({TF_SIZE}명) 미만인 스킬:')
    for _, r in shortage.iterrows():
        print(f'   - {r["TF스킬"]}: 후보 보유자 {r["후보보유자"]}명')

### 카드 2 선택 — 공급 희귀도를 적합도에 반영할 TF 스킬

In [ ]:
cb_kss = {}; rows_kss = []
for _, row in df_tf_supply.sort_values('공급희귀도', ascending=False).iterrows():
    s = row['TF스킬']
    label = (f"{s}  │  공급희귀도={row['공급희귀도']:.3f}  │  "
             f"후보보유자={row['후보보유자']}명({row['후보보유율(%)']}%)")
    cb = widgets.Checkbox(value=False, description=label,
                          layout=widgets.Layout(width='560px'),
                          style={'description_width':'initial'})
    cb_kss[s] = cb; rows_kss.append(cb)

# TF 스킬 외 나머지도 선택 가능하도록 (적합도 계산 시 TF 스킬만 사용되지만 UI는 전체)
btn_all_tf = widgets.Button(description='TF 스킬 전체', button_style='warning', layout=widgets.Layout(width='130px'))
btn_clr2   = widgets.Button(description='전체 해제',    button_style='',        layout=widgets.Layout(width='120px'))
def sel_all_tf(_):
    for s,cb in cb_kss.items(): cb.value = True
def clr2(_):
    for cb in cb_kss.values(): cb.value=False
btn_all_tf.on_click(sel_all_tf); btn_clr2.on_click(clr2)
cnt2 = widgets.HTML('')
def upd2(change): cnt2.value = f'<b>{sum(cb.value for cb in cb_kss.values())}개 선택됨</b>'
for cb in cb_kss.values(): cb.observe(upd2, names='value')
display(widgets.VBox([
    widgets.HTML('<b>공급 희귀도를 적합도에 반영할 스킬 선택 (I_kss=1)</b><br>'),
    widgets.HBox([btn_all_tf, btn_clr2, cnt2]),
    widgets.VBox(rows_kss, layout=widgets.Layout(height='200px', overflow_y='scroll',
                                                   border='1px solid #ccc', padding='6px'))]))

## EDA — 카드 3 : 스킬 난이도 (전체 기준)

In [ ]:
TOP_N = 15
top_hard = df_diff.head(TOP_N); top_easy = df_diff.tail(TOP_N).iloc[::-1]
fig, axes = plt.subplots(1, 2, figsize=(16, 6))
fig.suptitle('카드 3 : 스킬 난이도 (전체 기준)', fontsize=13, fontweight='bold', y=1.01)
for ax, df_sub, title, col in [
    (axes[0], top_hard, f'어려운 스킬 TOP {TOP_N}', '#8E44AD'),
    (axes[1], top_easy, f'쉬운 스킬 TOP {TOP_N}',   '#27AE60'),
]:
    bars = ax.barh(df_sub['스킬'], df_sub['난이도'], color=col, alpha=0.75)
    for bar, row in zip(bars, df_sub.itertuples()):
        ax.text(bar.get_width()+0.005, bar.get_y()+bar.get_height()/2,
                f'lv4+:{row.레벨4이상}/{row.전체보유자}명', va='center', fontsize=8)
    ax.set_xlim(0,1.2); ax.set_xlabel('난이도'); ax.set_title(title, fontsize=11); ax.invert_yaxis()
    for label in ax.get_yticklabels():
        if label.get_text() in TF_SKILLS:
            label.set_color('#f59e0b'); label.set_fontweight('bold')
plt.tight_layout(); plt.show()

### 카드 3 선택 — 과제 난이도 계산에 포함할 스킬

In [ ]:
cb_diff = {}; rows_diff = []
for _, row in df_diff.iterrows():
    s = row['스킬']
    is_tf = '⭐TF필수' if s in TF_SKILLS else ''
    label = f"{s}  │  난이도={row['난이도']:.3f}  │  lv4+:{row['레벨4이상']}/{row['전체보유자']}명  {is_tf}"
    cb = widgets.Checkbox(value=False, description=label,
                          layout=widgets.Layout(width='560px'),
                          style={'description_width':'initial'})
    cb_diff[s] = cb; rows_diff.append(cb)

btn_top20 = widgets.Button(description='어려운 스킬 TOP20', button_style='warning', layout=widgets.Layout(width='160px'))
btn_tf3   = widgets.Button(description='TF 스킬 선택',      button_style='warning', layout=widgets.Layout(width='130px'))
btn_clr3  = widgets.Button(description='전체 해제',          button_style='',        layout=widgets.Layout(width='120px'))
def sel_top20(_):
    top20 = set(df_diff.head(20)['스킬'])
    for s,cb in cb_diff.items(): cb.value = s in top20
def sel_tf3(_):
    for s,cb in cb_diff.items(): cb.value = s in TF_SKILLS
def clr3(_):
    for cb in cb_diff.values(): cb.value=False
btn_top20.on_click(sel_top20); btn_tf3.on_click(sel_tf3); btn_clr3.on_click(clr3)
cnt3 = widgets.HTML('')
def upd3(change): cnt3.value = f'<b>{sum(cb.value for cb in cb_diff.values())}개 선택됨</b>'
for cb in cb_diff.values(): cb.observe(upd3, names='value')
display(widgets.VBox([
    widgets.HTML('<b>과제 난이도 계산에 포함할 스킬 선택</b><br>⭐ = TF 필수 스킬<br>'),
    widgets.HBox([btn_top20, btn_tf3, btn_clr3, cnt3]),
    widgets.VBox(rows_diff, layout=widgets.Layout(height='300px', overflow_y='scroll',
                                                   border='1px solid #ccc', padding='6px'))]))

## EDA — 카드 4 : 인재 유형 분류 (차출 후보 기준, 인사이트 전용)

In [ ]:
talent_rows = []
for emp_id, row in cand_wide.iterrows():
    skills_held = row[row>0]
    n = len(skills_held)
    avg = skills_held.mean() if n>0 else 0
    std = skills_held.std()  if n>1 else 0
    talent_rows.append({'사번':emp_id,'보유스킬수':n,'평균레벨':avg,'레벨분산':std})
df_talent = pd.DataFrame(talent_rows)
med_n   = df_talent['보유스킬수'].median()
med_std = df_talent['레벨분산'].median()
def classify(r):
    if r['보유스킬수']<=med_n and r['레벨분산']>=med_std: return '전문가형'
    elif r['보유스킬수']>med_n and r['레벨분산']<med_std: return '제너럴리스트형'
    else: return '혼합형'
df_talent['유형'] = df_talent.apply(classify, axis=1)
type_colors = {'전문가형':'#2E86AB','제너럴리스트형':'#E84855','혼합형':'#F4A261'}
type_counts = df_talent['유형'].value_counts()
colors_sc   = [type_colors[t] for t in df_talent['유형']]

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle('카드 4 : 인재 유형 분류 (차출 후보 기준)', fontsize=13, fontweight='bold')
axes[0].scatter(df_talent['보유스킬수'], df_talent['레벨분산'], c=colors_sc, alpha=0.5, s=40)
axes[0].axvline(med_n,   color='gray', linestyle='--', alpha=0.6, label=f'스킬수 중앙값({med_n:.0f})')
axes[0].axhline(med_std, color='gray', linestyle=':',  alpha=0.6, label=f'분산 중앙값({med_std:.2f})')
for t,c in type_colors.items(): axes[0].scatter([],[],c=c,label=t,s=60)
axes[0].set_xlabel('보유 스킬 수'); axes[0].set_ylabel('레벨 표준편차'); axes[0].legend(fontsize=9)
axes[0].set_title('보유 스킬 수 vs 레벨 분산')
axes[1].pie(type_counts.values,
            labels=[f"{k}\n({v}명, {v/N_CAND*100:.1f}%)" for k,v in type_counts.items()],
            colors=[type_colors[k] for k in type_counts.index], startangle=90)
axes[1].set_title('차출 후보 유형 분포')
plt.tight_layout(); plt.show()
print('\n[유형별 평균 스킬 현황]')
print(df_talent.groupby('유형')[['보유스킬수','평균레벨','레벨분산']].mean().round(2))

## 적합도 매트릭스 산출 (차출 후보 × P21)
> **카드 1·2·3 선택 완료 후 실행하세요.**

In [ ]:
selected_idf  = [s for s,cb in cb_idf.items()  if cb.value]
selected_kss  = [s for s,cb in cb_kss.items()  if cb.value]
selected_diff = [s for s,cb in cb_diff.items() if cb.value]

print(f'카드 1 선택: {len(selected_idf)}개 | 카드 2 선택: {len(selected_kss)}개 | 카드 3 선택: {len(selected_diff)}개')

# TF 과제 난이도 (P21 기준)
tf_req_in_diff = [s for s in selected_diff if s in TF_SKILLS]
tf_difficulty  = sum(difficulty_raw.get(s,0) for s in tf_req_in_diff)
# 단일 과제라 정규화는 1.0 고정 (max = tf_difficulty itself)
tf_diff_norm   = 1.0 if tf_difficulty > 0 else 0.0
print(f'P21 난이도: {tf_difficulty:.3f} (정규화: {tf_diff_norm})')

# 카드 2 공급 희귀도 → kss_norm 대용
supply_rare_norm = {s: df_tf_supply[df_tf_supply['TF스킬']==s]['공급희귀도'].iloc[0]
                    if s in df_tf_supply['TF스킬'].values else 0.0
                    for s in ALL_SKILLS}

# 스킬 중요도
W3 = 0.5
skill_imp = {}
for s in ALL_SKILLS:
    skill_imp[s] = (1
                    + (1 if s in selected_idf else 0) * idf_norm.get(s, 0)
                    + (1 if s in selected_kss else 0) * supply_rare_norm.get(s, 0))

# 적합도 계산 (TF 필수 스킬만)
fit_series = pd.Series(index=candidate_ids, dtype=float)
for emp_id in candidate_ids:
    req_in = [s for s in TF_SKILLS if s in cand_wide.columns]
    imp    = np.array([skill_imp[s] for s in req_in])
    lvl    = cand_wide.loc[emp_id, req_in].values if emp_id in cand_wide.index else np.zeros(len(req_in))
    base   = (lvl * imp).sum()
    fit_series[emp_id] = base * (1 + W3 * tf_diff_norm)

print(f'\n적합도 산출 완료: {N_CAND}명')
print(f'최고 적합도: {fit_series.max():.2f} | 평균: {fit_series.mean():.2f} | 최저: {fit_series.min():.2f}')
print(f'\nTOP 10 후보:')
top10 = fit_series.nlargest(10)
id_to_name = df_hr.set_index('사번').apply(lambda r: f"{r['Last Name']}{r['First Name']}", axis=1).to_dict()
for emp_id, score in top10.items():
    proj = df_hr[df_hr['사번']==emp_id]['소속과제명'].iloc[0]
    print(f'  {emp_id} {id_to_name.get(emp_id,"")} ({proj}): {score:.2f}')

## 3단계 — 공백 방지 옵션 + ILP 최적 TF 구성

**흐름**
```
① 1차 ILP  → 순수 적합도 기반 TF 구성 (소프트 제약 없음)
② 자동 계산 → 1차 결과 기반 λ 자동 산출 + 평균 레벨 분포 표시
③ 사용자 설정 → AVG_LEVEL 슬라이더 + 소프트 제약 토글
④ 2차 ILP  → 소프트 제약 + 자동 λ → 최종 TF 구성
```

### 3-0. 기본 준비

In [ ]:
import pulp, time

cand_list = candidate_ids
n_c       = len(cand_list)
id_to_idx = {emp_id: i for i, emp_id in enumerate(cand_list)}

F_vec = np.array([fit_series.get(emp_id, 0) for emp_id in cand_list])
fem_c = np.array([(df_hr[df_hr['사번']==emp_id]['성별']=='여').values[0]
                   for emp_id in cand_list], dtype=float)
rk_c  = np.array([df_hr[df_hr['사번']==emp_id]['직급수'].values[0]
                   for emp_id in cand_list], dtype=float)

HOLD_LEVEL  = 1
MIN_REMAIN  = 1
TIME_LIMIT  = 300
GAP         = 0.01

print(f'차출 후보: {n_c}명 | TF 목표: {TF_SIZE}명')
print(f'차출 허용 팀: {allowed_projects}')
print(f'팀별 최대 차출: {MAX_OUT}')

### 3-1. 1차 ILP — 순수 적합도 기반 TF 구성 (소프트 제약 없음)

In [ ]:
print("1차 ILP 실행 중...")
t0 = time.time()
prob1 = pulp.LpProblem('tf1', pulp.LpMaximize)
x1 = {i: pulp.LpVariable(f'x1_{i}', cat='Binary') for i in range(n_c)}
obj1 = pulp.lpSum(F_vec[i]*x1[i] for i in range(n_c))

# 하드 ① TF 인원 수 == TF_SIZE
prob1 += pulp.lpSum(x1[i] for i in range(n_c)) == TF_SIZE
# 하드 ② 팀별 차출 상한
for p in allowed_projects:
    p_idxs = [id_to_idx[e] for e in team_members[p] if e in id_to_idx]
    prob1 += pulp.lpSum(x1[i] for i in p_idxs) <= MAX_OUT[p]

prob1 += obj1
solver1, sname1 = None, ""
for _try in ["HiGHS_API","HiGHS_CMD","CBC"]:
    try:
        if _try=="HiGHS_API":   cand=pulp.HiGHS(msg=False,timeLimit=TIME_LIMIT,gapRel=GAP); cn="HiGHS(API)"
        elif _try=="HiGHS_CMD": cand=pulp.HiGHS_CMD(msg=False,timeLimit=TIME_LIMIT,gapRel=GAP); cn="HiGHS(CMD)"
        else:                   cand=pulp.PULP_CBC_CMD(msg=False,timeLimit=TIME_LIMIT,gapRel=GAP); cn="CBC"
        if cand.available(): solver1,sname1=cand,cn; break
    except Exception: continue
if solver1 is None:
    solver1=pulp.PULP_CBC_CMD(msg=False,timeLimit=TIME_LIMIT,gapRel=GAP); sname1="CBC"

prob1.solve(solver1)
t1 = time.time()

tf1_members = [cand_list[i] for i in range(n_c) if x1[i].value() and x1[i].value()>0.5]
tf1_fit = sum(F_vec[i] for i in range(n_c) if x1[i].value() and x1[i].value()>0.5)
# 팀별 차출 현황 (1차)
team_removed_1st = {p: [e for e in tf1_members if e in team_members[p]] for p in allowed_projects}

print(f"1차 ILP 완료 | {sname1} | {pulp.LpStatus[prob1.status]} | {t1-t0:.1f}s")
print(f"TF 구성 {len(tf1_members)}명 | Fit={tf1_fit:.1f}")

### 3-2. λ 자동 계산 (데이터 기반)

In [ ]:
print("λ 자동 계산 중...")

# ── LAM_COV: TF 필수스킬 최악 교환비 ────────────────────────────────────────
# "TF 필수스킬 보유자를 차출하고 미보유자를 넣었을 때 최대 이득"
max_ratio = 0.0
for s in TF_SKILLS:
    if s not in df_wide.columns: continue
    h_idx  = [i for i in range(n_c) if df_wide.loc[cand_list[i], s] >= HOLD_LEVEL
              if cand_list[i] in df_wide.index]
    nh_idx = [i for i in range(n_c) if df_wide.loc[cand_list[i], s] < HOLD_LEVEL
              if cand_list[i] in df_wide.index]
    if not h_idx or not nh_idx: continue
    fit_A = np.array([F_vec[i] for i in h_idx])
    fit_B = np.array([F_vec[i] for i in nh_idx])
    lv_A  = np.array([float(df_wide.loc[cand_list[i], s]) for i in h_idx])
    gain  = fit_B.max() - fit_A.min()
    lv    = float(lv_A[fit_A.argmin()])
    if lv > 0 and gain > 0:
        max_ratio = max(max_ratio, gain/lv)
LAM_COV = max_ratio + 1e-6

# ── LAM_GENDER / LAM_RANK: 차출 후 기존 팀 기준 교환 시뮬레이션 ──────────────
# 1차 TF 구성 후 남은 기존 팀 기준으로 교환 비용 측정
g_costs, r_costs = [], []
for p in allowed_projects:
    removed    = team_removed_1st[p]
    remaining  = [e for e in team_members[p] if e not in removed]
    if not remaining: continue

    # 잔류 팀 fit = 잔류 인원의 스킬 레벨 합산
    remain_wide = df_wide.loc[df_wide.index.isin(remaining)]
    req_s = proj_skill_bin.index[proj_skill_bin[p]==1].tolist() if p in proj_skill_bin.columns else []
    team_fit = sum(remain_wide[s].sum() if s in remain_wide.columns else 0 for s in req_s)
    if team_fit == 0: continue

    n_remain = len(remaining)
    # 성별: 잔류 팀 불균형 여부
    hr_remain = df_hr[df_hr['사번'].isin(remaining)]
    n_f_remain = (hr_remain['성별']=='여').sum()
    if n_f_remain < pool_f * n_remain:
        # 잔류 팀 남성 ↔ 차출 후보 여성 교환 비용
        males_remain   = [e for e in remaining if (df_hr[df_hr['사번']==e]['성별']=='여').values[0]==False]
        females_cand   = [e for e in cand_list if (df_hr[df_hr['사번']==e]['성별']=='여').values[0]==True
                          and e not in tf1_members]
        for m in males_remain[:10]:
            m_lv = sum(df_wide.loc[m, s] if s in df_wide.columns else 0 for s in req_s) if m in df_wide.index else 0
            for f in females_cand[:20]:
                f_lv = sum(df_wide.loc[f, s] if s in df_wide.columns else 0 for s in req_s) if f in df_wide.index else 0
                g_costs.append((m_lv - f_lv) / team_fit)

    # 직급: 잔류 팀 불균형 여부
    avg_rk_remain = hr_remain['직급수'].mean() if len(hr_remain) > 0 else pool_r
    if abs(avg_rk_remain - pool_r) >= 0.3:
        if avg_rk_remain > pool_r:
            hi_remain = [e for e in remaining if df_hr[df_hr['사번']==e]['직급수'].values[0] > pool_r]
            lo_cand   = [e for e in cand_list if df_hr[df_hr['사번']==e]['직급수'].values[0] < pool_r
                         and e not in tf1_members]
            for h in hi_remain[:10]:
                h_lv = sum(df_wide.loc[h,s] if s in df_wide.columns else 0 for s in req_s) if h in df_wide.index else 0
                for l in lo_cand[:20]:
                    l_lv = sum(df_wide.loc[l,s] if s in df_wide.columns else 0 for s in req_s) if l in df_wide.index else 0
                    r_costs.append((h_lv - l_lv) / team_fit)
        else:
            lo_remain = [e for e in remaining if df_hr[df_hr['사번']==e]['직급수'].values[0] < pool_r]
            hi_cand   = [e for e in cand_list if df_hr[df_hr['사번']==e]['직급수'].values[0] > pool_r
                         and e not in tf1_members]
            for l in lo_remain[:10]:
                l_lv = sum(df_wide.loc[l,s] if s in df_wide.columns else 0 for s in req_s) if l in df_wide.index else 0
                for h in hi_cand[:20]:
                    h_lv = sum(df_wide.loc[h,s] if s in df_wide.columns else 0 for s in req_s) if h in df_wide.index else 0
                    r_costs.append((l_lv - h_lv) / team_fit)

avg_ts        = n_c / len(allowed_projects) if allowed_projects else 1
fit_mean_team = tf1_fit / len(allowed_projects) if allowed_projects else tf1_fit
g_rate = float(np.median(g_costs)) if g_costs else 0.05
r_rate = float(np.median(r_costs)) if r_costs else 0.05
LAM_GENDER = fit_mean_team * g_rate / (avg_ts * pool_f) if pool_f > 0 else 2.7
LAM_RANK   = fit_mean_team * r_rate / rk_c.std() if rk_c.std() > 0 else 2.7

print(f"  LAM_COV    = {LAM_COV:.2f}")
print(f"  LAM_GENDER = {LAM_GENDER:.2f}  (교환 비용 중앙값 {g_rate*100:.2f}%)")
print(f"  LAM_RANK   = {LAM_RANK:.2f}  (교환 비용 중앙값 {r_rate*100:.2f}%)")
print(f"  교환 샘플: 성별 {len(g_costs)}개 | 직급 {len(r_costs)}개")
print("λ 자동 계산 완료")

### 3-3. 평균 레벨 분포 확인 + 배치 옵션 설정

In [ ]:
# ── 1차 TF 차출 후 기존 팀별 평균 레벨 분포 ─────────────────────────────────
team_avg_levels = []
for p in allowed_projects:
    removed   = team_removed_1st[p]
    remaining = [e for e in team_members[p] if e not in removed]
    if not remaining: continue
    req_s = proj_skill_bin.index[proj_skill_bin[p]==1].tolist() if p in proj_skill_bin.columns else []
    remain_wide = df_wide.loc[df_wide.index.isin(remaining)]
    levels = []
    for s in req_s:
        lv_sum = remain_wide[s].sum() if s in remain_wide.columns else 0
        if len(remaining) > 0:
            levels.append(lv_sum / len(remaining))
    team_avg_levels.append({
        '과제': p, '팀평균레벨': round(np.mean(levels), 2) if levels else 0,
        '잔류인원': len(remaining), '차출인원': len(removed)
    })

lv_df   = pd.DataFrame(team_avg_levels).sort_values('팀평균레벨')
lv_min  = lv_df['팀평균레벨'].min()
lv_max  = lv_df['팀평균레벨'].max()
lv_mean = lv_df['팀평균레벨'].mean()

fig, ax = plt.subplots(figsize=(14, 4))
colors = ['#ef4444' if v < lv_mean else '#6366f1' for v in lv_df['팀평균레벨']]
ax.bar(lv_df['과제'], lv_df['팀평균레벨'], color=colors, alpha=0.8)
ax.axhline(lv_mean, color='gray', linestyle='--', alpha=0.7, label=f'전체 평균 {lv_mean:.2f}')
ax.set_title('1차 TF 차출 후 기존 팀별 필수스킬 평균 레벨 (잔류 인원 기준)', fontsize=12, fontweight='bold')
ax.set_xlabel('과제'); ax.set_ylabel('평균 레벨')
ax.legend(); plt.xticks(rotation=45); plt.tight_layout(); plt.show()

print(f"차출 후 팀별 평균 레벨 범위: {lv_min:.2f} ~ {lv_max:.2f}  (전체 평균 {lv_mean:.2f})")
print(f"→ 아래 슬라이더에서 AVG_LEVEL을 설정하세요.")
print(f"   낮게 설정할수록 느슨한 기준, 높게 설정할수록 엄격한 기준입니다.")

In [ ]:
# ── 사용자 설정 UI ────────────────────────────────────────────────────────────

# 보유 커버리지
toggle_cov = widgets.ToggleButton(
    value=True, description='보유 커버리지 ON', button_style='success',
    layout=widgets.Layout(width='200px'))
def on_cov(change):
    toggle_cov.description = '보유 커버리지 ON' if change['new'] else '보유 커버리지 OFF'
    toggle_cov.button_style = 'success' if change['new'] else ''
toggle_cov.observe(on_cov, names='value')

# 평균 레벨 유지
toggle_avg = widgets.ToggleButton(
    value=True, description='평균 레벨 유지 ON', button_style='success',
    layout=widgets.Layout(width='200px'))
def on_avg(change):
    toggle_avg.description = '평균 레벨 유지 ON' if change['new'] else '평균 레벨 유지 OFF'
    toggle_avg.button_style = 'success' if change['new'] else ''
    slider_avg.disabled = not change['new']
toggle_avg.observe(on_avg, names='value')

slider_avg = widgets.FloatSlider(
    value=round(lv_mean, 1), min=round(lv_min, 1), max=round(lv_max, 1),
    step=0.1, description='AVG_LEVEL:',
    style={'description_width': 'initial'},
    layout=widgets.Layout(width='400px'),
    readout_format='.1f')

# 성별 균형 유지
toggle_gender = widgets.ToggleButton(
    value=True, description='성별 균형 유지 ON', button_style='success',
    layout=widgets.Layout(width='200px'))
def on_gender(change):
    toggle_gender.description = '성별 균형 유지 ON' if change['new'] else '성별 균형 유지 OFF'
    toggle_gender.button_style = 'success' if change['new'] else ''
toggle_gender.observe(on_gender, names='value')

# 직급 균형 유지
toggle_rank = widgets.ToggleButton(
    value=True, description='직급 균형 유지 ON', button_style='success',
    layout=widgets.Layout(width='200px'))
def on_rank(change):
    toggle_rank.description = '직급 균형 유지 ON' if change['new'] else '직급 균형 유지 OFF'
    toggle_rank.button_style = 'success' if change['new'] else ''
toggle_rank.observe(on_rank, names='value')

lam_info = widgets.HTML(
    f'<div style="background:#f0f0f0;padding:10px;border-radius:6px;font-size:12px">'
    f'<b>자동 계산된 λ (데이터 기반)</b><br>'
    f'LAM_COV = {LAM_COV:.2f} | '
    f'LAM_GENDER = {LAM_GENDER:.2f} | '
    f'LAM_RANK = {LAM_RANK:.2f}<br>'
    f'<small>※ 차출 후 기존 팀 균형 유지 기준으로 계산</small>'
    f'</div>')

display(widgets.VBox([
    widgets.HTML('<b>공백 방지 옵션 설정</b><br><br>'),
    lam_info,
    widgets.HTML('<br><b>소프트 제약 온오프</b>'),
    widgets.HBox([toggle_cov, toggle_avg, toggle_gender, toggle_rank]),
    widgets.HTML('<br><b>차출 후 기존 팀 평균 레벨 기준값</b> (1차 TF 차출 결과 분포 참고)'),
    slider_avg,
    widgets.HTML(f'<small>범위: {lv_min:.2f} (최저팀) ~ {lv_max:.2f} (최고팀) | 전체평균: {lv_mean:.2f}</small>')
]))

### 3-4. 보유 커버리지 사전 검사
> **위 3-3 설정 완료 후 실행하세요.**

In [ ]:
ENABLE_SKILL_COV = toggle_cov.value
ENABLE_AVG_LEVEL = toggle_avg.value
ENABLE_GENDER    = toggle_gender.value
ENABLE_RANK      = toggle_rank.value
AVG_LEVEL        = slider_avg.value

print(f"[설정 확인]")
print(f"  보유 커버리지: {ENABLE_SKILL_COV}")
print(f"  평균 레벨 유지: {ENABLE_AVG_LEVEL} | AVG_LEVEL = {AVG_LEVEL:.1f}")
print(f"  성별/직급 균형 유지: {ENABLE_GENDER}/{ENABLE_RANK}")
print(f"  LAM_COV={LAM_COV:.2f} | LAM_GENDER={LAM_GENDER:.2f} | LAM_RANK={LAM_RANK:.2f}")

infeasible = []
if ENABLE_SKILL_COV:
    for p in allowed_projects:
        req_s = proj_skill_bin.index[proj_skill_bin[p]==1].tolist() if p in proj_skill_bin.columns else []
        mbr_wide_p = df_wide.loc[df_wide.index.isin(team_members[p])]
        for s in req_s:
            total_hold = team_skill_hold.get((p,s), 0)
            if total_hold < MIN_REMAIN: continue
            if s in mbr_wide_p.columns:
                hold_idxs = [id_to_idx[e] for e in team_members[p]
                             if e in id_to_idx and mbr_wide_p.loc[e,s] >= HOLD_LEVEL]
            else:
                hold_idxs = []
            if hold_idxs and total_hold - MAX_OUT[p] < MIN_REMAIN:
                infeasible.append((p, s, total_hold))

if infeasible:
    print('\n⚠️  공백 위험 스킬:')
    for p,s,h in infeasible:
        print(f'   {p} / {s}: 보유자 {h}명, 최대 차출 {MAX_OUT[p]}명 → 잔류 위험')
    print('\n→ ENABLE_SKILL_COV를 OFF하거나 MAX_OUT을 줄이세요.')
else:
    print('\n✅ 보유 커버리지 사전 검사 통과')

### 3-5. 2차 ILP — 최종 TF 구성

In [ ]:
t0 = time.time()
prob2 = pulp.LpProblem('tf2', pulp.LpMaximize)
x2 = {i: pulp.LpVariable(f'x2_{i}', cat='Binary') for i in range(n_c)}
obj2 = pulp.lpSum(F_vec[i]*x2[i] for i in range(n_c))

# ── 하드 ① TF 인원 수 == TF_SIZE ─────────────────────────────────────────────
prob2 += pulp.lpSum(x2[i] for i in range(n_c)) == TF_SIZE

# ── 하드 ② 팀별 차출 상한 ────────────────────────────────────────────────────
for p in allowed_projects:
    p_idxs = [id_to_idx[e] for e in team_members[p] if e in id_to_idx]
    prob2 += pulp.lpSum(x2[i] for i in p_idxs) <= MAX_OUT[p]

# ── 하드 ③ 보유 커버리지: 차출 후 기존 팀 필수 스킬 보유자 ≥ MIN_REMAIN ─────────
if ENABLE_SKILL_COV:
    for p in allowed_projects:
        req_s = proj_skill_bin.index[proj_skill_bin[p]==1].tolist() if p in proj_skill_bin.columns else []
        mbr_wide_p = df_wide.loc[df_wide.index.isin(team_members[p])]
        for s in req_s:
            total_hold = team_skill_hold.get((p,s), 0)
            if total_hold < MIN_REMAIN: continue
            if s in mbr_wide_p.columns:
                hold_idxs = [id_to_idx[e] for e in team_members[p]
                             if e in id_to_idx and mbr_wide_p.loc[e,s] >= HOLD_LEVEL]
            else:
                hold_idxs = []
            if hold_idxs:
                prob2 += pulp.lpSum(x2[i] for i in hold_idxs) <= total_hold - MIN_REMAIN

# ── 소프트 ① 평균 레벨 유지: 차출 후 기존 팀 레벨 기준 ──────────────────────
avg_slack = {}
if ENABLE_AVG_LEVEL:
    for p in allowed_projects:
        req_s = proj_skill_bin.index[proj_skill_bin[p]==1].tolist() if p in proj_skill_bin.columns else []
        p_idxs = [id_to_idx[e] for e in team_members[p] if e in id_to_idx]
        for s in req_s:
            sl = pulp.LpVariable(f'avg_{p}_{s}', lowBound=0)
            avg_slack[(p,s)] = sl
            fixed_lv     = team_skill_sum.get((p,s), 0)
            remove_lv    = pulp.lpSum(
                (float(df_wide.loc[cand_list[i],s]) if s in df_wide.columns and cand_list[i] in df_wide.index else 0)
                * x2[i] for i in p_idxs)
            remove_cnt   = pulp.lpSum(x2[i] for i in p_idxs)
            prob2 += (fixed_lv - remove_lv + sl
                      >= AVG_LEVEL * Nj[p] - AVG_LEVEL * remove_cnt)
            obj2 -= LAM_COV * sl

# ── 소프트 ② 성별 균형 유지: 차출 후 기존 팀 기준 ───────────────────────────
if ENABLE_GENDER:
    for p in allowed_projects:
        dp = pulp.LpVariable(f'gdp_{p}', lowBound=0)
        dm = pulp.LpVariable(f'gdm_{p}', lowBound=0)
        p_idxs = [id_to_idx[e] for e in team_members[p] if e in id_to_idx]
        fixed_f    = float(team_fem_count[p])
        remove_f   = pulp.lpSum(fem_c[i]*x2[i] for i in p_idxs)
        remove_cnt = pulp.lpSum(x2[i] for i in p_idxs)
        prob2 += (fixed_f - remove_f - pool_f*Nj[p] + pool_f*remove_cnt == dp-dm)
        obj2 -= LAM_GENDER*(dp+dm)

# ── 소프트 ③ 직급 균형 유지: 차출 후 기존 팀 기준 ───────────────────────────
if ENABLE_RANK:
    for p in allowed_projects:
        dp = pulp.LpVariable(f'rdp_{p}', lowBound=0)
        dm = pulp.LpVariable(f'rdm_{p}', lowBound=0)
        p_idxs = [id_to_idx[e] for e in team_members[p] if e in id_to_idx]
        fixed_r    = float(team_rank_sum[p])
        remove_r   = pulp.lpSum(rk_c[i]*x2[i] for i in p_idxs)
        remove_cnt = pulp.lpSum(x2[i] for i in p_idxs)
        prob2 += (fixed_r - remove_r - pool_r*Nj[p] + pool_r*remove_cnt == dp-dm)
        obj2 -= LAM_RANK*(dp+dm)

prob2 += obj2
build_t = time.time()-t0
print(f'모델 구성 완료 | {build_t:.1f}s | 변수 {len(prob2.variables()):,} | 제약 {len(prob2.constraints):,}')

### 3-6. 풀이

In [ ]:
solver2, sname2 = None, ""
for _try in ["HiGHS_API","HiGHS_CMD","CBC"]:
    try:
        if _try=="HiGHS_API":   cand=pulp.HiGHS(msg=True,timeLimit=TIME_LIMIT,gapRel=GAP); cn="HiGHS(API)"
        elif _try=="HiGHS_CMD": cand=pulp.HiGHS_CMD(msg=True,timeLimit=TIME_LIMIT,gapRel=GAP); cn="HiGHS(CMD)"
        else:                   cand=pulp.PULP_CBC_CMD(msg=True,timeLimit=TIME_LIMIT,gapRel=GAP); cn="CBC"
        if cand.available(): solver2,sname2=cand,cn; break
    except Exception: continue
if solver2 is None:
    solver2=pulp.PULP_CBC_CMD(msg=True,timeLimit=TIME_LIMIT,gapRel=GAP); sname2="CBC"

print(f'선택된 솔버: {sname2}')
t1=time.time(); prob2.solve(solver2); solve_t=time.time()-t1
print(f'\n솔버 {sname2} | 상태 {pulp.LpStatus[prob2.status]} | 풀이 {solve_t:.1f}s')

if pulp.LpStatus[prob2.status]=='Infeasible':
    print('\n❌ 해가 없습니다.')
    print('   → 보유 커버리지를 OFF하거나 TF 목표 인원을 줄이거나 MAX_OUT을 늘리세요.')

## 결과 — TF 구성 + 기존 팀 공백 검증

In [ ]:
tf_members = [cand_list[i] for i in range(n_c) if x2[i].value() and x2[i].value()>0.5]
tf_fit     = sum(F_vec[i] for i in range(n_c) if x2[i].value() and x2[i].value()>0.5)
unmet      = sum(1 for v in avg_slack.values() if v.value() and v.value()>1e-6) if avg_slack else 0

print('='*60)
print(f'솔버 {sname2} | 상태 {pulp.LpStatus[prob2.status]}')
print(f'TF 구성 {len(tf_members)}명 / 목표 {TF_SIZE}명')
print(f'1차 Fit {tf1_fit:.1f} → 최종 Fit {tf_fit:.1f}')
if ENABLE_AVG_LEVEL:
    print(f'평균 레벨({AVG_LEVEL:.1f}) 미달: {unmet}/{len(avg_slack)}건')
print('='*60)

# TF 멤버 상세
print('\n[TF 구성원]')
tf_detail = []
for emp_id in sorted(tf_members, key=lambda e: -F_vec[cand_list.index(e)]):
    hr_row = df_hr[df_hr['사번']==emp_id].iloc[0]
    tf_skills_held = [s for s in TF_SKILLS if s in df_wide.columns and df_wide.loc[emp_id,s]>0
                      if emp_id in df_wide.index]
    tf_detail.append({
        '사번': emp_id,
        '성명': id_to_name.get(emp_id,''),
        '소속과제': hr_row['소속과제명'],
        '직위': hr_row['직위'],
        '성별': hr_row['성별'],
        'P21 적합도': round(F_vec[cand_list.index(emp_id)],2),
        'TF스킬보유': ', '.join(tf_skills_held) if tf_skills_held else '없음'
    })
display(pd.DataFrame(tf_detail))

tf_hr  = df_hr[df_hr['사번'].isin(tf_members)]
tf_fem = (tf_hr['성별']=='여').mean()
tf_rk  = tf_hr['직급수'].mean()
print(f'\nTF 성비: 여성 {tf_fem*100:.1f}% | TF 평균 직급: {tf_rk:.2f}')

In [ ]:
# 기존 팀 공백 검증
print('\n[기존 팀 공백 검증]')
gap_results = []
for p in allowed_projects:
    removed   = [e for e in tf_members if e in team_members[p]]
    if not removed: continue
    remaining = [e for e in team_members[p] if e not in removed]
    n_remain  = len(remaining)
    req_s     = proj_skill_bin.index[proj_skill_bin[p]==1].tolist() if p in proj_skill_bin.columns else []

    # 스킬 커버리지 검사
    skill_gaps = []
    for s in req_s:
        hold_before = team_skill_hold.get((p,s),0)
        removed_hold = sum(1 for e in removed if s in df_wide.columns
                           and e in df_wide.index and df_wide.loc[e,s]>=HOLD_LEVEL)
        if hold_before - removed_hold < MIN_REMAIN:
            skill_gaps.append(f'{s}({hold_before-removed_hold}명 잔류)')

    # 평균 레벨 검사
    lv_issues = []
    for s in req_s:
        lv_before = team_skill_sum.get((p,s),0)
        lv_removed = sum(df_wide.loc[e,s] if s in df_wide.columns and e in df_wide.index else 0
                         for e in removed)
        lv_after = lv_before - lv_removed
        if n_remain > 0 and lv_after/n_remain < AVG_LEVEL:
            lv_issues.append(f'{s}({lv_after/n_remain:.1f}<{AVG_LEVEL})')

    removed_names = [f"{id_to_name.get(e,e)}({df_hr[df_hr['사번']==e]['직위'].iloc[0]})"
                     for e in removed]
    gap_results.append({
        '과제':p, '차출인원':len(removed), '잔류인원':n_remain,
        '차출된 인원': ', '.join(removed_names),
        '스킬커버리지': '✅' if not skill_gaps else f'⚠️ {", ".join(skill_gaps)}',
        '평균레벨': '✅' if not lv_issues else f'⚠️ {", ".join(lv_issues)}'
    })

if gap_results:
    gap_df = pd.DataFrame(gap_results)
    display(gap_df)
    skill_viol = sum(1 for r in gap_results if '⚠️' in r['스킬커버리지'])
    lv_viol    = sum(1 for r in gap_results if '⚠️' in r['평균레벨'])
    print(f'\n스킬 커버리지 위반: {skill_viol}개 팀 | 평균 레벨 미달: {lv_viol}개 팀')
    if skill_viol==0 and lv_viol==0:
        print('✅ 모든 기존 팀 공백 없음')
else:
    print('차출 인원 없음')